# Module 04: Multi-Node JAX GPU Training with JobSet
In this module, you will launch and inspect a multi-node JAX GPU training workload on GKE using Kubernetes JobSet.

### Key Concepts Covered:
1. **Multi-Host JAX Initialization**: `jax.distributed.initialize()` with `COORDINATOR_ADDRESS` and rank indexing over Headless DNS.
2. **GPU Inter-Node All-Reduce**: `jax.lax.psum` across GPU nodes via NCCL.
3. **SPMD Mesh Sharding**: Sharding JAX tensors across multi-node GPU memories using `jax.sharding.Mesh`.



In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
import config
cfg = config.load_config("../config.env")

PROJECT_ID = cfg["PROJECT_ID"]
REGION = cfg["REGION"]
REPO = cfg["ARTIFACT_REGISTRY_REPO"]
GPU_IMAGE = cfg["GPU_IMAGE_NAME"]
TAG = cfg["IMAGE_TAG"]

GPU_FULL_IMAGE = f"{REGION}-docker.pkg.dev/{PROJECT_ID}/{REPO}/{GPU_IMAGE}:{TAG}"
print(f"Target GPU Image: {GPU_FULL_IMAGE}")



## 1. Render GPU JobSet Manifest


In [ ]:
manifest_path = Path("../manifests/jobset-gpu.yaml")
rendered_path = Path("../manifests/jobset-gpu-rendered.yaml")

with open(manifest_path, "r") as f:
    content = f.read()

content = content.replace("LOCATION-docker.pkg.dev/PROJECT_ID/ARTIFACT_REGISTRY_REPO/GPU_IMAGE_NAME:IMAGE_TAG", GPU_FULL_IMAGE)

with open(rendered_path, "w") as f:
    f.write(content)

print(f"Rendered JobSet manifest saved to {rendered_path}")



## 2. Deploy JAX GPU Multi-Node JobSet


In [ ]:
!kubectl apply -f ../manifests/jobset-gpu-rendered.yaml


## 3. Monitor JobSet & Worker Pod Status


In [ ]:
import time
print("Waiting for GPU worker pods to initialize...")
for i in range(8):
    !kubectl get jobset jax-gpu-job
    !kubectl get pods -l jobset.x-k8s.io/jobset-name=jax-gpu-job -o custom-columns=POD_NAME:.metadata.name,NODE:.spec.nodeName,STATUS:.status.phase
    time.sleep(5)



## 4. Stream Logs from Multi-Node GPU Workers


In [ ]:
!kubectl logs -l jobset.x-k8s.io/jobset-name=jax-gpu-job --all-containers --tail=100


## 5. Verify GPU Execution & Mathematical Sum Proof
Inspect the logs to confirm:
- `JAX Distributed Initialized Successfully!` across all GPU ranks.
- GPU NCCL `lax.psum` output equals `3.0`.
